# Person 1 — Product metadata collection

**Question:** Which product characteristics are associated with average Amazon customer ratings?

Source: [McAuley Lab Amazon Reviews 2023](https://amazon-reviews-2023.github.io/),
distributed through [Hugging Face](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023).
The overall review corpus spans May 1996–September 2023. We collect product metadata;
our subset has no review dates, so we do not claim this as its observed date range.

The four categories give us numeric variables (rating, rating count, raw price) and
categorical variables (source category, main category, store). One row is one product
entry within a category; uniqueness across categories must be checked during cleaning.
All collection logic lives in `src.collect`, not in this notebook.

This notebook verifies Parker Fabian's (Person 1) raw metadata collection. It checks the pinned source, inspects raw structure and reconciles category counts before handing data to Michael for cleaning. All collection logic remains in `src.collect`; the notebook does not clean or impute values.

## 1. Locate the project and pinned source

Resolve the repository from either supported starting directory, then import Parker's shared collection module. The printed revision identifies the reproducible source snapshot.

In [1]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir():
    root = root.parent
if not (root / "src" / "collect.py").is_file():
    raise RuntimeError("Open this notebook from the project root or notebooks folder")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import pandas as pd
import pyarrow.parquet as pq
from src.collect import collect_to_disk, load_category, row_counts, REVISION

print("Source revision:", REVISION)

Source revision: 2b6d039ed471f2ba5fd2acb718bf33b0a7e5598e


C:\Users\Parker\Documents\Codex\2026-09-25\files-mentioned-by-the-user-team-2\work\miniconda3\envs\amazon-reviews\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Obtain the full raw table

The shared collector streams the ten source shards, omits media columns and adds the source `category` tag while preserving raw nested types. Reuse an existing collected file to avoid replacing it during notebook verification; otherwise create it with the shared streaming collector.

The footer below reports the **full saved table's** shape, without loading its long text columns into memory.

In [2]:
output_path = root / "data" / "raw" / "products_raw.parquet"
if output_path.is_file():
    print("Reusing existing collected raw table")
else:
    output_path = collect_to_disk()
raw_file = pq.ParquetFile(output_path)
print("Saved:", output_path.relative_to(root))
print("Full raw shape:", (raw_file.metadata.num_rows, len(raw_file.schema_arrow.names)))

Reusing existing collected raw table
Saved: data\raw\products_raw.parquet
Full raw shape: (1644621, 14)


## 3. Inspect raw values and structure

Load only the first five All_Beauty entries with the shared loader to inspect raw field representations. This preview is not random and is not a statistical sample of the project.

In [3]:
preview = load_category("All_Beauty", nrows=5)
preview.head()

All_Beauty: 5 rows


,main_category,title,average_rating,rating_number,features,description,price,store,categories,details,parent_asin,subtitle,author,category
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],None,Howard Products,[],"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ...",B01CUPMQZE,None,None,All_Beauty
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],None,Yes To,[],"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro...",B076WQZGPM,None,None,All_Beauty
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],None,Levine Health Products,[],"{""Manufacturer"": ""Levine Health Products""}",B000B658RI,None,None,All_Beauty
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],None,Cherioll,[],"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""...",B088FKY3VD,None,None,All_Beauty
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,None,Precision,[],"{""UPC"": ""644287689178""}",B07NGFDN6G,None,None,All_Beauty


Inspect the stored Arrow schema for the full file. This makes the raw list and string fields visible without converting or cleaning them.

In [4]:
print(raw_file.schema_arrow)

main_category: string
title: string
average_rating: double
rating_number: int64
features: list<element: string>
  child 0, element: string
description: list<element: string>
  child 0, element: string
price: string
store: string
categories: list<element: string>
  child 0, element: string
details: string
parent_asin: string
subtitle: string
author: string
category: string
-- schema metadata --
huggingface: '{"info": {"features": {"main_category": {"dtype": "string",' + 1231


## 4. Verify completeness and the collection contract

Read only the category column from the saved table for a memory-conscious audit. These are actual saved row counts, not preview counts or numbers copied from a prior submission.

In [5]:
counts = pd.read_parquet(output_path, columns=["category"])["category"].value_counts().sort_index()
counts.rename("rows").to_frame()

,rows
category,
All_Beauty,112590
Industrial_and_Scientific,427564
Musical_Instruments,213593
Toys_and_Games,890874


Compare saved category counts with pinned source parquet footers and check that required fields are present and omitted media fields are absent. These checks verify collection, not deduplication or cleaning.

In [6]:
expected = row_counts()
assert counts.to_dict() == dict(sorted(expected.items()))
assert int(counts.sum()) == raw_file.metadata.num_rows
assert not {"images", "videos", "bought_together"}.intersection(raw_file.schema_arrow.names)
assert {"parent_asin", "average_rating", "rating_number", "price", "main_category", "store", "category"}.issubset(raw_file.schema_arrow.names)
print("All source rows accounted for:", sum(expected.values()))

All source rows accounted for: 1644621


## 5. Handoff to Person 2

Run `python -m src.collect` from the repository root, then use
`from src.collect import load_raw; df = load_raw()` in your cleaning notebook.
The file is `data/raw/products_raw.parquet` and is intentionally excluded from Git.

Inspect the actual raw price and category representations before converting them.
Use `parent_asin` to investigate duplicates; distinguish the source `category` tag
from `main_category`. A store name is not necessarily a brand. Missing values and
duplicate entries have deliberately not been altered.

This notebook verifies collection, not statistical findings. Person 2 owns cleaning,
Person 3 owns summary statistics, and Person 4 owns charts. Price/rating associations
cannot by themselves establish causation.